# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants
MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("OpenAI API Key was correctly setup.")
else:
    print("OpenAPI API Key is missing.")

openai = OpenAI()

technical_explainer_system_prompt = """
You will be asked code technical questions.
You will be provided with snippets of code that can be
the body of a function or the body of class,
explain every line and what it does,
remembering the context from the previous lines.
Respond in markdown without html tags.
"""

def messages_for(system_prompt, user_question):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]


In [ ]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = soup.find_all("a")
    links = [link.get("href") for link in links if link.get("href")]
    return links
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming
def stream_technical_answer(user_question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages_for(technical_explainer_system_prompt, user_question),
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

stream_technical_answer(question)

In [ ]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages_for(technical_explainer_system_prompt, question))
technical_answer = response.choices[0].message.content

display(Markdown(technical_answer))